In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
print("Imported successfully")

In [ ]:
# load dataset
df = pd.read_csv(r"D:\Project_programming\case_study_mc\Data\processesed\hse2_data.csv")
print("Loaded successfully")

In [ ]:
# global parameters
sns.set_style('darkgrid')
plt.rcParams.update({
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.edgecolor':'#444444',
    'axes.labelcolor':'#333333',
    'xtick.color':'#333333',
    'ytick.color':'#333333',
})

### Figure 1:

In [ ]:
# overall smoking prevalence
plot_data = (
    df.groupby("year")["current_smoker"]
       .mean()
       .mul(100)
       .reset_index()
)
fig, ax = plt.subplots(figsize = (12,7))
sns.barplot(
    data=plot_data,
    x="year",
    y="current_smoker",
    ax = ax,
    edgecolor = 'black',
    color = '#D97706'
)
ax.set_title(
  "Current Smoking Prevalence: 1999 vs 2019",
  fontsize = 14,
  fontweight = 'bold'
)
ax.set_xlabel(
  "Survey Year",
  fontsize = 12,
  fontweight = 'bold'
)
ax.set_ylabel(
  "Current smokers (%)",
  fontsize = 12,
  fontweight = 'bold'
)


### Figure 2

In [ ]:
# smoking prevalence by gender
gender_trend = (
    df.groupby(["year", "sex"])
       .agg(
            sample_size=("current_smoker", "count"),
            smokers=("current_smoker", "sum"),
            gender_prevalence=("current_smoker", "mean")
       )
       .reset_index()
)

gender_trend["gender_prevalence"] *= 100
gender_trend["sex"] = gender_trend["sex"].map({
    1.0: "Male",
    2.0: "Female"
})
fig, ax = plt.subplots(figsize=(12, 7))
sns.barplot(
    data=gender_trend,
    x="year",
    y="gender_prevalence",
    hue="sex",
    ax=ax
)
ax.set_title(
    "Smoking Prevalence by Gender",
    fontsize=14,
    fontweight="bold"
)
ax.set_xlabel(
    "Survey Year",
    fontsize=12,
    fontweight="bold"
)
ax.set_ylabel(
    "Current smokers (%)",
    fontsize=12,
    fontweight="bold"
)
ax.legend(title="Sex")


### Figure 3

In [ ]:
# smoking prevalence by region
regional = (
    df.groupby(["year", "region"])
       ["current_smoker"]
       .mean()
       .mul(100)
       .reset_index(name="prevalence")
)
regional['region'] = regional['region'].map({
  1.0:"North East",
  2.0:"North West",
  3.0:"Yorkshire and The Humber",
  4.0:"East Midlands",
  5.0:"West Midlands",
  6.0:"East of England",
  7.0:"London",
  8.0:"South East",
  9.0:"South West"
})
fig, ax = plt.subplots(figsize = (12,7))

sns.barplot(
    data=regional,
    x="region",
    y="prevalence",
    hue="year",
    ax = ax
)
ax.set_title(
    "Smoking Prevalence by Region",
    fontsize=14,
    fontweight="bold"
)
ax.set_xlabel(
    "Government Office Region",
    fontsize=12,
    fontweight="bold"
)
ax.set_ylabel(
    "Current smokers (%)",
    fontsize=12,
    fontweight="bold"
)
ax.legend(title = 'Year')
plt.setp(ax.get_xticklabels(), rotation=45, ha="right");

### Figure 4:

In [ ]:
# Predicted Probability of Current Smoking by Gender and Survey Year

# put year as a categorical variable
df['year'] = df['year'].astype(str)

# remove NaN values and convert the current smoker to int
df = df.dropna(subset = ['current_smoker']).copy()
df['current_smoker'] = df['current_smoker'].astype(int)

# Logistic regression model
gender_model = smf.logit(
  'current_smoker ~ C(year)* C(sex)',
  data = df
).fit()
# Create prediction dataset
prediction_data = pd.DataFrame({
    'year': ['1999', '2019', '1999', '2019'],
    'sex': [1.0, 1.0, 2.0, 2.0]
})

# Generate predicted probabilities
prediction_data['predicted_probability'] = gender_model.predict(
    prediction_data
)

# Convert probabilities into percentages
prediction_data['predicted_percentage'] = (
    prediction_data['predicted_probability'] * 100
)

print(prediction_data)

In [ ]:
fig, ax = plt.subplots(figsize = (12,7))

sns.lineplot(
    data=prediction_data,
    ax = ax,
    x='year',
    y='predicted_percentage',
    hue='sex',
    marker='o',
    markersize=10,
    linewidth=2.5
)

plt.xticks(['1999', '2019'])
ax.set_title(
    "Predicted Smoking Prevalence by Gender (1999–2019)",
    fontsize=14,
    fontweight="bold"
)
ax.set_xlabel(
    "Survey Year",
    fontsize=12,
    fontweight="bold"
)
ax.set_ylabel(
    "Predicted Probability of Current Smoking (%)",
    fontsize=12,
    fontweight="bold"
)
ax.legend(title = 'Gender',labels = ['Male', 'Female'])

